In [1]:
pip install shared_utils

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importing necessary package 
import pandas as pd 
import geopandas as gpd
import google.auth
import os
import gcsfs
from calitp_data_analysis.sql import get_engine
from calitp_data_analysis import utils
db_engine = get_engine()
credentials, project = google.auth.default()
fs = gcsfs.GCSFileSystem()

pd.set_option('display.max_columns', None)

/opt/conda/lib/python3.11/site-packages/dask/dataframe/__init__.py:31: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


In [3]:
GCS_FILE_PATH  = 'gs://calitp-analytics-data/data-analyses/ahsc_grant/ahsc_riderships'

In [34]:
bart_ridership = pd.read_excel(f"{GCS_FILE_PATH}/bart_ridership.xlsx")
big_blue_bus_ridership = pd.read_excel(f"{GCS_FILE_PATH}/big_blue_bus_ridership.xlsx")
caltrain_ridership = pd.read_excel(f"{GCS_FILE_PATH}/caltrain_ridership.xlsx")

In [25]:
def normalize_day_type(
    df,
    day_type_col,
    day_of_week_col
):
    """
    Reclassifies 'Holiday' rows into Weekday/Saturday/Sunday
    based on the day of week.
    """
    return df.apply(
        lambda row: (
            'Weekday'
            if row[day_type_col] == 'Holiday'
            and row[day_of_week_col] in ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
            else (
                'Saturday'
                if row[day_type_col] == 'Holiday'
                and row[day_of_week_col] == 'Saturday'
                else (
                    'Sunday'
                    if row[day_type_col] == 'Holiday'
                    and row[day_of_week_col] == 'Sunday'
                    else row[day_type_col]
                )
            )
        ),
        axis=1
    )

In [ ]:
# Function to count number of days of a Day_Type in a service period
def count_day_type_days(start, end, day_type):
    dates = pd.date_range(start, end)
    day_type_upper = day_type.upper()
    if day_type_upper == 'WEEKDAY':
        return sum(d.weekday() < 5 for d in dates)  
    elif day_type_upper == 'SATURDAY':
        return sum(d.weekday() == 5 for d in dates)
    elif day_type_upper == 'SUNDAY':
        return sum(d.weekday() == 6 for d in dates)
    else:
        return 0

In [26]:
bart_ridership['Day Type'] = normalize_day_type(
    bart_ridership,
    day_type_col='Day Type',
    day_of_week_col='Day of Week'
)

bart_ridership['exposure'] = bart_ridership.apply(
    lambda row: count_day_type_days(row['start_date'], row['end_date'], row['Day Type']),
    axis=1
)

bart_ridership.rename(columns={
    'Station': 'Stop',
    'Station Name': 'Stop Name',
    'Entries': 'Boardings'
}, inplace=True)

In [11]:
# Compute exposure: number of days of that Day_Type in the period
big_blue_bus_ridership['exposure'] = big_blue_bus_ridership.apply(
    lambda row: count_day_type_days(row['start_date'], row['end_date'], row['SERVICE_DAY']),
    axis=1
)

# Compute total boardings for the period
big_blue_bus_ridership['Boardings'] = big_blue_bus_ridership['AVERAGE_DAILY_BOARDINGS'] * big_blue_bus_ridership['exposure']

# Rename columns
big_blue_bus_ridership.rename(columns={
    'SERVICE_DAY': 'Day_Type',
    'STOP_ID': 'Stop',
    'STOP_NAME': 'Stop Name'
}, inplace=True)


For Caltrain Ridership, dropping "Holiday" data since we just have average ridership for the month. 

In [30]:
caltrain_ridership.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2520 entries, 0 to 2519
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Month               2520 non-null   object        
 1   Origin Station      2520 non-null   object        
 2   Caltrain Ridership  2520 non-null   float64       
 3   Date Type           2520 non-null   object        
 4   Average Ridership   2520 non-null   float64       
 5   start_date          2520 non-null   datetime64[ns]
 6   end_date            2520 non-null   datetime64[ns]
dtypes: datetime64[ns](2), float64(2), object(3)
memory usage: 137.9+ KB


In [36]:
caltrain_ridership = caltrain_ridership[caltrain_ridership['Date Type'] != 'Holiday']


# Compute exposure: number of days of that Day_Type in the period
caltrain_ridership['exposure'] = caltrain_ridership.apply(
    lambda row: count_day_type_days(row['start_date'], row['end_date'], row['Date Type']),
    axis=1
)

# Compute total boardings for the period (count for PPML)
caltrain_ridership['Boardings'] = caltrain_ridership['Average Ridership'] * caltrain_ridership['exposure']

In [38]:
caltrain_ridership.sample(5)

,Month,Origin Station,Caltrain Ridership,Date Type,Average Ridership,start_date,end_date,exposure,Boardings
1415,February 2025,Burlingame,13751.353936,Sunday,275.440074,2025-02-01,2025-02-28,4,1101.760295
854,December 2024,Millbrae,30264.742770,Saturday,657.510808,2024-12-01,2024-12-31,4,2630.043232
1188,January 2024,Redwood City,31812.744251,Saturday,451.499473,2024-01-01,2024-01-31,4,1805.997893
457,April 2024,Capitol,738.126214,Weekday,33.551192,2024-04-01,2024-04-30,22,738.126214
1098,April 2024,Redwood City,40015.899267,Saturday,536.137195,2024-04-01,2024-04-30,4,2144.548779
